# 01 — Veri Kontrolü, Bölünme ve Development-only EDA
**Ders:** Yapay Sinir Ağları — Yüksek Lisans Ara Sınav Projesi
**Veri:** UCI Bank Marketing — yalnızca `bank-additional.csv` (4.119 gözlem, 20 girdi, 1 hedef)

Bu notebook'un sırası (Ödev Bölüm 3–4):
1. Seed'lerin türetilmesi
2. Ham verinin okunması ve doğrulanması (değişken adları, tipler, hedef etiketleri)
3. `duration` kararının sabitlenmesi (model performansına bakılmadan alınan uygulama kuralı)
4. Kopya kayıt kontrolü
5. Stratified %80 development / %20 test bölünmesi → `results/dataset_split.csv`
6. **Yalnızca development** üzerinde EDA ve deney hipotezleri

In [ ]:
# === Kurulum: Google Colab (Drive'da ortak klasör) veya yerel Jupyter ===
import os, sys, subprocess
GITHUB_REPO_URL = "https://github.com/ilkinurcan/ysa-mlp-bank-marketing-ilkin.git"
DRIVE_DIR = "/content/drive/MyDrive/YSA_odev"   # üç notebook aynı klasörü kullanır

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    repo_dir = os.path.join(DRIVE_DIR, GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git"))
    if not os.path.exists(os.path.join(repo_dir, "src")):
        subprocess.run(["git", "clone", GITHUB_REPO_URL, repo_dir], check=True)
    else:  # repo'daki kod güncellemelerini al (üretilen sonuç dosyalarına dokunmaz)
        r = subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], capture_output=True, text=True)
        print("git pull:", (r.stdout or r.stderr).strip()[:300])
    os.chdir(repo_dir)
elif not os.path.exists("src"):
    raise RuntimeError("Repo klasöründe değilsin: os.chdir('<repo klasörü>') yap.")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Çalışma klasörü:", os.getcwd())

In [ ]:
import numpy as np, pandas as pd
from IPython.display import display
from src import config, data, plots
from src.config import NUMERIC, CATEGORICAL, RESULTS_DIR, FIG_DIR

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
seeds = config.get_seeds()
config.set_global_seeds(seeds)
config.save_json(seeds, config.CONFIG_DIR / "seeds.json")
env = config.write_environment_files()
print("Temel seed = son 5 hane + 2026 =", seeds["base_seed"])
display(pd.DataFrame([(k, v) for k, v in seeds.items()], columns=["seed", "değer"]))
env

## 1. Ham veri ve doğrulama
Veri UCI arşivinden indirilir; zip içinden **yalnızca** `bank-additional.csv` çıkarılır, `bank-additional-full.csv` kullanılmaz.

In [ ]:
data.download_dataset()
raw = data.load_raw()
checks = data.validate_raw(raw)
config.save_json(checks, RESULTS_DIR / "raw_validation.json")
display(pd.Series(checks, name="kontrol"))
assert checks["all_ok"], "Veri beklenen şemada değil!"
display(raw.head())
display(raw.dtypes.to_frame("dtype").T)

## 2. `duration` kararı
`duration` (görüşme süresi) görüşme bitmeden bilinmez ve hedefle neredeyse belirleyici bir ilişki taşır (UCI veri açıklaması da gerçekçi bir model için çıkarılmasını önerir). Bu nedenle **model performansı incelenmeden önce** tüm modellerden çıkarılmıştır (`src/config.py → DROP_COLUMNS`).

Ayrıca hedeften bağımsız, satır-bazlı (fit gerektirmeyen) sabit bir kural uygulanır: `pdays = 999` "daha önce temas yok" anlamına geldiği için silinmez; `pdays_no_prev_contact` göstergesi eklenir ve `pdays` değeri bu satırlarda NaN yapılarak **kat-içinde** median ile doldurulur.

In [ ]:
dup = data.duplicate_report(raw)
config.save_json(dup, RESULTS_DIR / "duplicate_report.json")
pd.Series(dup, name="kopya kontrolü")

**Kopya yönetimi:** `duration` hariç tüm girdileri aynı olan satırlara ortak bir `duplicate_group` kimliği verilir. Bölünmeler bu grup üzerinden yapılır; böylece aynı (kopya) gözlem hem eğitim hem değerlendirme tarafına düşmez. Kopya yoksa yöntem sıradan `StratifiedKFold` / stratified split ile birebir aynıdır; kopya varsa `StratifiedGroupKFold` kullanılır (`src/data.py`).

In [ ]:
split, split_summary = data.create_split(seeds, test_size=0.20)
display(split_summary)
print(split["split"].value_counts())

---
## 3. EDA — yalnızca development (%80)
Aşağıdaki tüm hücreler `load_development_raw()` ile yalnızca development satırlarını kullanır. Test satırları bu notebook'ta hiçbir karşılaştırmaya girmez. Hedefe göre verilen tablolar **betimseldir**; bunlara dayanarak özellik eleme veya imputation seçimi yapılmaz.

In [ ]:
dev = data.load_development_raw()
y_dev = data.encode_target(dev)
dev_feat = dev.drop(columns=["duration"])
rows = []
for c in dev_feat.columns:
    s = dev_feat[c]
    rows.append({"özellik": c, "tip": "sayısal" if pd.api.types.is_numeric_dtype(s) else "kategorik",
                 "dtype": str(s.dtype), "benzersiz": s.nunique(), "NaN": int(s.isna().sum()),
                 "unknown": 0 if pd.api.types.is_numeric_dtype(s) else int((s.astype(str) == "unknown").sum()),
                 "özel kod": "999 = önceden temas yok" if c == "pdays" else ""})
schema = pd.DataFrame(rows)
schema["unknown_%"] = (100 * schema["unknown"] / len(dev)).round(2)
schema.to_csv(RESULTS_DIR / "eda_schema.csv", index=False)
print("Development boyutu:", dev.shape, "| girdi sayısı (duration hariç):", dev_feat.shape[1] - 1)
schema

In [ ]:
cls = y_dev.value_counts().rename_axis("y").to_frame("n")
cls["%"] = (100 * cls["n"] / cls["n"].sum()).round(2)
cls.index = cls.index.map({0: "no (0)", 1: "yes (1)"})
cls.to_csv(RESULTS_DIR / "eda_class_distribution.csv")
display(cls)
print(f"Dengesizlik oranı (neg/pos) = {cls['n'].iloc[0] / cls['n'].iloc[1]:.2f}")
print(f"Her şeye 'no' diyen modelin accuracy'si = {cls['%'].iloc[0]:.2f}% ; AP taban değeri (pozitif oranı) = {y_dev.mean():.4f}")
plots.class_distribution(y_dev.values)

**Metrik seçimine etkisi:** Pozitif sınıf azınlıktadır. Çoğunluk sınıfını tahmin eden bir model yüksek accuracy alır ama hiçbir pozitifi bulamaz. Bu yüzden birincil metrik **Average Precision (AP)** seçilmiştir. AP'nin rastgele modeldeki tabanı pozitif oranıdır. Sensitivity, Specificity, F1 ve Balanced Accuracy de ayrıca raporlanır.

In [ ]:
num = dev[NUMERIC].copy()
num["pdays"] = num["pdays"].where(num["pdays"] != 999)   # 999 özel kodu hariç tutularak
num = num.rename(columns={"pdays": "pdays (999 hariç)"})
desc = pd.DataFrame({
    "n": num.count(), "ortalama": num.mean(), "medyan": num.median(), "SD": num.std(ddof=1),
    "Q1": num.quantile(.25), "Q3": num.quantile(.75), "IQR": num.quantile(.75) - num.quantile(.25),
    "min": num.min(), "maks": num.max(), "çarpıklık": num.skew(),
}).round(3)
desc.to_csv(RESULTS_DIR / "eda_numeric_stats.csv")
desc

In [ ]:
plots.numeric_distributions(num, list(num.columns), y_dev)

In [ ]:
freq_rows = []
for c in CATEGORICAL:
    vc = dev[c].value_counts()
    for k, v in vc.items():
        freq_rows.append({"özellik": c, "kategori": k, "n": v, "%": round(100 * v / len(dev), 2),
                          "kategori içi yes %": round(100 * y_dev[dev[c] == k].mean(), 2)})
freq = pd.DataFrame(freq_rows)
freq.to_csv(RESULTS_DIR / "eda_categorical_freq.csv", index=False)
unk = freq[freq["kategori"] == "unknown"][["özellik", "n", "%", "kategori içi yes %"]]
unk.to_csv(RESULTS_DIR / "eda_unknown_codes.csv", index=False)
print("'unknown' kodları (development):"); display(unk)
freq.head(25)

In [ ]:
plots.categorical_bars(dev, ["job", "education", "default", "contact", "month", "poutcome"], y_dev)

In [ ]:
pd_tab = pd.DataFrame({
    "n": [(dev.pdays == 999).sum(), (dev.pdays != 999).sum()],
    "yes %": [100 * y_dev[dev.pdays == 999].mean(), 100 * y_dev[dev.pdays != 999].mean()],
}, index=["pdays = 999 (temas yok)", "pdays < 999 (önceden temas)"]).round(2)
pd_tab["%"] = (100 * pd_tab["n"] / len(dev)).round(2)
pd_tab.to_csv(RESULTS_DIR / "eda_pdays_999.csv")
display(pd_tab)
print("poutcome ile çapraz tablo:"); display(pd.crosstab(dev.poutcome, dev.pdays == 999, margins=True))

In [ ]:
corr_cols = [c for c in num.columns]
_, corr = plots.correlation_matrix(num, corr_cols)
corr.to_csv(RESULTS_DIR / "eda_correlation_pearson.csv")
pairs = (corr.where(np.triu(np.ones(corr.shape, bool), 1)).stack()
         .rename("r").reset_index().assign(abs_r=lambda d: d.r.abs()).sort_values("abs_r", ascending=False))
pairs.head(8)

In [ ]:
# Seçili sayısal özelliklerin hedefe göre betimsel karşılaştırması (seçim için KULLANILMAZ)
by_y = num.groupby(y_dev.values).agg(["mean", "median"]).T.unstack()
by_y.columns = [f"y={a}_{b}" for a, b in by_y.columns]
by_y.round(3).to_csv(RESULTS_DIR / "eda_numeric_by_target.csv")
by_y.round(3)

In [ ]:
# Aykırı değer işaretleme: IQR kuralı ve %1/%99 yüzdelikleri (silme yapılmaz)
out_rows = []
for c in num.columns:
    s = num[c].dropna(); q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    out_rows.append({"özellik": c, "IQR_alt": lo, "IQR_üst": hi, "alt_aykırı_n": int((s < lo).sum()),
                     "üst_aykırı_n": int((s > hi).sum()), "aykırı_%": round(100 * ((s < lo) | (s > hi)).mean(), 2),
                     "P1": s.quantile(.01), "P99": s.quantile(.99), "P99_üstü_n": int((s > s.quantile(.99)).sum())})
outl = pd.DataFrame(out_rows).round(3)
outl.to_csv(RESULTS_DIR / "eda_outliers.csv", index=False)
outl

## 4. EDA bulgularından ön işleme önerileri
- **Aykırı değerler:** `campaign`, `previous`, `age` gibi sağa çarpık değişkenlerde IQR dışı gözlemler vardır. Bunlar gerçek müşteri davranışıdır, veri hatası değildir. Bu yüzden **silinmez**. Alternatif olarak eğitim katında öğrenilen %1–%99 yüzdelik **clipping** ve aykırı değerlere dayanıklı **RobustScaler** arama uzayına konmuştur.
- **`pdays = 999`:** Development satırlarının büyük çoğunluğunda bu kod vardır. Silinmez; gösterge değişken + kat-içi median doldurma ile kodlanır.
- **`unknown`:** Özellikle `default` sütununda yüksek orandadır. Varsayılan olarak **ayrı kategori** tutulur. "En sık sınıfla doldurma" alternatifi, ek analizde her outer-train içinde inner CV ile yeniden seçilerek değerlendirilir (02 notebook).
- **Korelasyon:** Makroekonomik değişkenler (`emp.var.rate`, `euribor3m`, `nr.employed`) birbirleriyle çok yüksek korelasyonludur. Etkin bilgi boyutu düşüktür. Hedefe dayalı özellik eleme **yapılmaz**; düzenleme (weight decay, dropout) arama uzayındadır.
- **Ölçekleme:** Değişken aralıkları çok farklıdır (ör. `nr.employed` ~5000, `emp.var.rate` ~[-3,1]). MLP'nin gradyan tabanlı eğitimi için ölçekleme zorunludur. **Standardization** (StandardScaler, z-skoru: ortalama 0, SD 1), **normalization** (MinMaxScaler, [0,1] aralığı) ve RobustScaler (median/IQR) karşılaştırılır.

## 5. Önceden belirlenmiş deney hipotezleri (test edilebilir)
Bu hipotezler 02 notebook'u çalıştırılmadan önce yazılmıştır ve sonuçlara göre değiştirilmeyecektir.

| Kod | Hipotez | Test edildiği karşılaştırma |
|---|---|---|
| **H1** | Sınıf dengesizliği nedeniyle varsayılan 0.50 eşiğinde MLP'nin **Sensitivity** değeri **Specificity**'den belirgin biçimde düşük olacaktır. Dummy model yüksek accuracy alsa da AP'si pozitif oranı düzeyinde kalacaktır. | Outer CV: Sensitivity vs Specificity; Dummy vs MLP AP |
| **H2** | Aynı LR=1e-3 ile **SGD+Momentum (M3)**, Adam (M2) ve AdamW'ye (M1) göre daha yavaş yakınsayacaktır: en iyi epoch daha geç gelecek / 150 epoch sınırına daha sık ulaşılacaktır. İç CV AP'si eşit veya daha düşük olacaktır. | M1 vs M2 vs M3: iç CV AP, best epoch, süre |
| **H3** | Çarpık ve aykırı değerli sayısal değişkenler nedeniyle **MinMaxScaler (M4)**, StandardScaler'a (M1) göre eşit veya daha düşük iç CV AP verecektir. | M1 vs M4 |
| **H4** | Makro değişkenlerin yüksek korelasyonu nedeniyle etkin boyut düşüktür. **(64,32)** mimarisi (M5), (32,) mimarisine (M1) iç CV SD'sinden büyük bir AP üstünlüğü sağlamayacaktır. | M1 vs M5 (fark / SD) |
| **H5** | **ReduceLROnPlateau kapatıldığında (M6)** iç early-stop kaybı daha erken platoya girecek ve iç CV AP'si M1'e eşit veya daha düşük olacaktır. | M1 vs M6: AP, best epoch |
| **H6** | Küçük ve tablosal veri setinde ayarlanmış MLP, sabit ön işlemeli **Logistic Regression**'ı outer CV AP bakımından kat-arası SD'yi aşan bir farkla geçmeyecektir. | Outer CV: MLP vs LogisticRegression |